# NOTEBOOK: 06 — Gold Aggregation

- **Layer:** Gold
- **Purpose:** Produce a business-ready aggregated summary from Judge-approved records only.
- **Inputs:** `main.techmart_silver.taxonomy_enriched`
- **Outputs:** `main.techmart_gold.product_summary`
- **Notes:** Only `judge_approved = True` records aggregated. Grouped by Category and Sub-Category with num_products, avg, min and max price.
- **Author:** Maira Tavares

# 0. Imports and Configuration

In [0]:
from pyspark.sql import functions as F

In [0]:
from utils.config import (
    TAXONOMY_ENRICHED,
    GOLD_SUMMARY
)

In [0]:

print("=" * 55)
print("NOTEBOOK 06 — GOLD AGGREGATION")
print("=" * 55)
print(f"  {'Source':<20} : {TAXONOMY_ENRICHED}")
print(f"  {'Target':<20} : {GOLD_SUMMARY}")
print("=" * 55)
print("✅ Ready to run")

# 1. Build Gold Aggregation

In [0]:
# Build the Gold aggregation table
# This is the business-ready table that analysts and dashboards consume
# Aggregated by Category and Sub-Category

df_gold = (
    spark.table(TAXONOMY_ENRICHED)
    .filter(F.col("judge_approved") == "True")  # ← only approved records
    .groupBy("category", "sub_category")
    .agg(
        F.count("product_id").alias("num_products"),
        F.round(F.avg("unit_price_usd"), 2).alias("avg_price_usd"),
        F.round(F.min("unit_price_usd"), 2).alias("min_price_usd"),
        F.round(F.max("unit_price_usd"), 2).alias("max_price_usd")
    )
    .orderBy("category", "sub_category")
)

print("✅ Gold aggregation complete")
display(df_gold)

# 2. Save and Validate

In [0]:
# Write to Gold Delta table
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    # .option("overwriteSchema", "true")
    .saveAsTable(GOLD_SUMMARY))

print(f"✅ Written: {GOLD_SUMMARY}")
print(f"   Rows    : {spark.table(GOLD_SUMMARY).count()}")

In [0]:
# Validation summary
df_summary = spark.table(GOLD_SUMMARY)

print("=" * 55)
print("GOLD AGGREGATION — VALIDATION SUMMARY")
print("=" * 55)
print(f"  {'Total categories':<25} : {df_summary.select('category').distinct().count()}")
print(f"  {'Total sub-categories':<25} : {df_summary.count()}")
print(f"  {'Total products':<25} : {df_summary.agg(F.sum('num_products')).collect()[0][0]}")
print(f"  {'Overall avg price':<25} : ${df_summary.agg(F.round(F.avg('avg_price_usd'), 2)).collect()[0][0]}")
print(f"  {'Target table':<25} : {GOLD_SUMMARY}")
print("=" * 55)
print("✅ Gold aggregation complete")

display(df_summary)